# Stacking and blending
The heart-disease data (303 patients, 13 inputs, `target` 1 = disease). Three base models (random forest, KNN,
gradient boosting) and a logistic regression meta-model: first with scikit-learn's `StackingClassifier`, then
blending and K-fold stacking built by hand, so every step is visible.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier,
                              StackingClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. The data and the split

In [2]:
# 303 patients; 'target' is 1 for heart disease, 0 for none
df = pd.read_csv("data/heart.csv")
print(df.shape)
print(df["target"].value_counts())

# inputs and output
X = df.drop(columns=["target"])
y = df["target"]

# 80% to train (242 patients), 20% to test (61 patients)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)
print(X_train.shape, X_test.shape)

(303, 14)
target
1    165
0    138
Name: count, dtype: int64
(242, 13) (61, 13)


## 2. The base models alone

In [3]:
def base_models(scale_knn=False):
    """The three base models; KNN optionally inside a scaling pipeline."""
    knn = KNeighborsClassifier(n_neighbors=10)
    if scale_knn:
        # KNN measures distances, so put every column on mean 0, std 1 first
        knn = make_pipeline(StandardScaler(), knn)
    return [
        ("rf", RandomForestClassifier(n_estimators=10, random_state=42)),
        ("knn", knn),
        ("gbdt", GradientBoostingClassifier(random_state=42)),
    ]

# each base model trained on the training set and scored on the test set
for name, model in base_models():
    model.fit(X_train, y_train)
    print(name, round(accuracy_score(y_test, model.predict(X_test)), 3))

rf 0.787
knn 0.639


gbdt 0.852


## 3. StackingClassifier

In [4]:
# base models -> 10-fold out-of-fold predictions -> logistic regression meta-model
clf = StackingClassifier(
    estimators=base_models(),
    final_estimator=LogisticRegression(max_iter=1000),
    cv=10,
)
clf.fit(X_train, y_train)
print("stacking accuracy:", round(clf.score(X_test, y_test), 3))

# the meta-model's weights: how much it trusts each base model's probability
print(dict(zip(["rf", "knn", "gbdt"], clf.final_estimator_.coef_[0].round(2))))

stacking accuracy: 0.869
{'rf': np.float64(2.31), 'knn': np.float64(0.59), 'gbdt': np.float64(1.75)}


In [5]:
# what the meta-model sees: one column per base model (probability of class 1)
print(clf.transform(X_test.iloc[:5]).round(2))

[[0.9  0.8  0.88]
 [0.3  0.5  0.83]
 [0.8  0.6  0.98]
 [0.1  0.1  0.01]
 [0.2  0.4  0.02]]


## 4. Scaling KNN, passthrough and soft voting

In [6]:
results = {}
for name, model in base_models(scale_knn=True):
    results[name] = accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))

# soft voting: average the three probabilities, no meta-model
vote = VotingClassifier(base_models(scale_knn=True), voting="soft").fit(X_train, y_train)
results["soft voting"] = vote.score(X_test, y_test)

# stacking with a scaled KNN
stack = StackingClassifier(base_models(scale_knn=True),
                           final_estimator=LogisticRegression(max_iter=1000), cv=10)
results["stacking"] = stack.fit(X_train, y_train).score(X_test, y_test)

# passthrough=True: the meta-model also gets the 13 original columns (scaled for logistic regression)
stack_pt = StackingClassifier(base_models(scale_knn=True),
                              final_estimator=make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
                              cv=10, passthrough=True)
results["stacking, passthrough"] = stack_pt.fit(X_train, y_train).score(X_test, y_test)
print(stack_pt.transform(X_test.iloc[:1]).shape, "meta-model inputs per row")

for k, v in results.items():
    print(f"{k:22s} {v:.3f}")

(1, 16) meta-model inputs per row
rf                     0.787
knn                    0.836
gbdt                   0.852
soft voting            0.885
stacking               0.869
stacking, passthrough  0.852


## 5. Blending by hand (hold-out)

In [7]:
# split the training set again: base models learn on 'fit', the meta-model on 'val'
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
print(len(X_fit), "rows for the base models,", len(X_val), "for the meta-model")

models = [m for _, m in base_models(scale_knn=True)]
for m in models:
    m.fit(X_fit, y_fit)

# new dataset: one column per base model, predicted on rows the base models never saw
val_preds = np.column_stack([m.predict_proba(X_val)[:, 1] for m in models])
meta = LogisticRegression().fit(val_preds, y_val)

# prediction: base models first, then the meta-model on their outputs
test_preds = np.column_stack([m.predict_proba(X_test)[:, 1] for m in models])
print("blending accuracy:", round(accuracy_score(y_test, meta.predict(test_preds)), 3))

193 rows for the base models, 49 for the meta-model


blending accuracy: 0.82


## 6. K-fold stacking by hand (out-of-fold predictions)

In [8]:
kf = KFold(n_splits=4, shuffle=True, random_state=42)

# step 1: for each base model, 4 fits; each fold is predicted by the model trained on the other 3
oof = np.column_stack([
    cross_val_predict(m, X_train, y_train, cv=kf, method="predict_proba")[:, 1]
    for _, m in base_models(scale_knn=True)
])
print(oof.shape, "= every training row, one column per base model")

# step 2: the meta-model learns from the out-of-fold predictions
meta = LogisticRegression().fit(oof, y_train)

# step 3: retrain every base model on the whole training set
final = [m.fit(X_train, y_train) for _, m in base_models(scale_knn=True)]

# prediction on the test set
test_preds = np.column_stack([m.predict_proba(X_test)[:, 1] for m in final])
print("K-fold stacking accuracy:", round(accuracy_score(y_test, meta.predict(test_preds)), 3))

(242, 3) = every training row, one column per base model


K-fold stacking accuracy: 0.869


## 7. Why not predict on the training data?

In [9]:
# the naive recipe: base models predict the very rows they were trained on
naive = np.column_stack([m.predict_proba(X_train)[:, 1] for m in final])
print("training-row predictions, mean |p - y|:")
print("  in-sample   ", np.abs(naive - y_train.values[:, None]).mean(axis=0).round(3))
print("  out-of-fold ", np.abs(oof - y_train.values[:, None]).mean(axis=0).round(3))
meta_naive = LogisticRegression().fit(naive, y_train)
print("naive meta weights      ", meta_naive.coef_[0].round(2))
print("out-of-fold meta weights", meta.coef_[0].round(2))
print("naive stacking accuracy:", round(accuracy_score(y_test, meta_naive.predict(test_preds)), 3))

training-row predictions, mean |p - y|:
  in-sample    [0.088 0.226 0.071]
  out-of-fold  [0.295 0.257 0.248]
naive meta weights       [3.52 0.96 3.81]
out-of-fold meta weights [1.34 2.8  0.97]
naive stacking accuracy: 0.852


## 8. Blending versus K-fold stacking when data is limited
One split with 61 test patients is too noisy to compare the two methods, so we repeat the comparison on 100 random
splits and average. Only the training size changes: 60, 120 or 242 patients for training, the rest for testing.
Both methods use the same base models and the same logistic regression meta-model; blending holds out 20% of the
training patients for the meta-model, stacking uses 5-fold out-of-fold predictions.

In [10]:
import warnings
warnings.filterwarnings("ignore")

rows = []
for n_train in (60, 120, 242):
    for seed in range(100):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, train_size=n_train, random_state=seed, stratify=y)
        # blending: base models on 80% of the training patients, meta-model on the other 20%
        X_f, X_v, y_f, y_v = train_test_split(X_tr, y_tr, test_size=0.2, random_state=seed, stratify=y_tr)
        ms = [m.fit(X_f, y_f) for _, m in base_models(scale_knn=True)]
        meta = LogisticRegression().fit(np.column_stack([m.predict_proba(X_v)[:, 1] for m in ms]), y_v)
        blend = accuracy_score(y_te, meta.predict(np.column_stack([m.predict_proba(X_te)[:, 1] for m in ms])))
        # K-fold stacking: meta-model on 5-fold out-of-fold predictions, base models refitted on all training patients
        stack = StackingClassifier(base_models(scale_knn=True), final_estimator=LogisticRegression(), cv=5)
        rows.append((n_train, seed, blend, stack.fit(X_tr, y_tr).score(X_te, y_te)))

res = pd.DataFrame(rows, columns=["n_train", "seed", "blending", "stacking"])
res.to_csv("data/blend_vs_stack.csv", index=False)
res["gain"] = res["stacking"] - res["blending"]
print(res.groupby("n_train").agg(blending=("blending", "mean"), stacking=("stacking", "mean"),
                                 gain=("gain", "mean"),
                                 gain_se=("gain", lambda g: g.std(ddof=1) / np.sqrt(len(g))),
                                 stacking_better=("gain", lambda g: int((g > 0).sum())),
                                 ties=("gain", lambda g: int((g == 0).sum()))).round(3))

         blending  stacking   gain  gain_se  stacking_better  ties
n_train                                                           
60          0.753     0.787  0.035    0.006               74     5
120         0.801     0.814  0.012    0.002               62     8
242         0.812     0.815  0.003    0.003               41    17


## 9. Out-of-fold predictions with K = 2, 4 and 10 (3,000 synthetic rows)

In [11]:
import numpy as np, warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import log_loss
# mismatch between out-of-fold predictions (what the meta-model learns from) and full-data models (what it receives)
X, y = make_classification(n_samples=3000, n_features=20, n_informative=8, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=1/3, random_state=0)
models = {"rf": RandomForestClassifier(n_estimators=50, random_state=0), "knn": KNeighborsClassifier(15),
          "gbdt": GradientBoostingClassifier(random_state=0)}
for name, m in models.items():
    full = log_loss(yte, m.fit(Xtr, ytr).predict_proba(Xte)[:, 1])
    row = []
    for K in (2, 4, 10):
        oof = cross_val_predict(m, Xtr, ytr, cv=KFold(K, shuffle=True, random_state=0), method="predict_proba")[:, 1]
        row.append(f"K={K}: {log_loss(ytr, oof):.3f}")
    print(f"{name:5s} out-of-fold log loss  " + "  ".join(row) + f"   | full-data model on test: {full:.3f}")


rf    out-of-fold log loss  K=2: 0.238  K=4: 0.221  K=10: 0.215   | full-data model on test: 0.222


knn   out-of-fold log loss  K=2: 0.232  K=4: 0.231  K=10: 0.223   | full-data model on test: 0.331


gbdt  out-of-fold log loss  K=2: 0.161  K=4: 0.154  K=10: 0.154   | full-data model on test: 0.169
